# Segmented WavLM — Comparison Notebook

Compares two feature representations on the **same** train/test split:
- **Whole-audio** — mean-pool full response → 768-dim (baseline)
- **Segmented** — 10s windows → mean + std pooled → 1536-dim (new)

Output CSVs (no clash with cheating_detection_v3):
- `{folder}_wavlm_whole.csv` — 768-dim, same format as v3's `_wavlm.csv` but different name
- `{folder}_wavlm_seg.csv`   — 1536-dim segmented


In [ ]:
# ================================================================
# CONFIGURATION — edit this cell only
# ================================================================
from pathlib import Path

# Training folders — add or comment out as needed
TRAIN_FOLDERS = [
    "audios2",
    "audios4",
]

# Test folder — leave "" to hold out TEST_RATIO of train automatically
TEST_FOLDER = ""   # e.g. "audios5"

# WavLM model: pretrained baseline OR local path to fine-tuned folder
WAVLM_MODEL_PATH = "microsoft/wavlm-base-plus"
# After Colab fine-tuning:
# WAVLM_MODEL_PATH = r"wavlm_finetuned"

WINDOW_SEC  = 10    # seconds per window
HOP_SEC     = 5     # hop between windows
MIN_WIN_SEC = 4     # discard windows shorter than this

SAVE_DIR    = "checkpoints_seg"
TEST_RATIO  = 0.20
RANDOM_SEED = 42
AUDIO_EXTS  = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
SR          = 16000

NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / SAVE_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os, warnings, json as _json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import xgboost as xgb
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, classification_report, confusion_matrix
)
from transformers import AutoFeatureExtractor, WavLMModel

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

LABEL_MAP = {
    "cheating": 1, "read": 1, "scripted": 1, "yes": 1, "1": 1,
    "not cheating": 0, "spontaneous": 0, "no": 0, "0": 0,
}


## 1. Scan Folders


In [ ]:
def scan_folder(folder_name):
    audio_dir = NB_DIR / folder_name
    if not audio_dir.exists():
        print(f'  SKIP {folder_name}: folder not found'); return None
    audio_files = sorted(f for f in audio_dir.rglob('*')
                         if f.suffix.lower() in AUDIO_EXTS)
    if not audio_files:
        print(f'  SKIP {folder_name}: no audio files'); return None
    gt_path = NB_DIR / f'{folder_name}GT.csv'
    if not gt_path.exists():
        print(f'  SKIP {folder_name}: no GT file'); return None
    return {
        "name":        folder_name,
        "audio_dir":   audio_dir,
        "audio_files": audio_files,
        "gt_path":     gt_path,
        "whole_csv":   NB_DIR / f"{folder_name}_wavlm_whole.csv",
        "seg_csv":     NB_DIR / f"{folder_name}_wavlm_seg.csv",
    }

all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))
folders   = [m for m in (scan_folder(n) for n in all_names) if m]

for m in folders:
    hw = 'done' if m['whole_csv'].exists() else 'needed'
    hs = 'done' if m['seg_csv'].exists()   else 'needed'
    print(f"{m['name']:12s}  {len(m['audio_files']):4d} files  "
          f"whole={hw}  seg={hs}")


## 2. Load WavLM Model

Loaded once, shared for both extraction passes.


In [ ]:
print(f'Loading WavLM from: {WAVLM_MODEL_PATH}')
_fe    = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL_PATH)
_model = WavLMModel.from_pretrained(WAVLM_MODEL_PATH).eval().to(DEVICE)
print('Loaded.')

def load_audio_16k(path):
    try:
        y, sr = sf.read(str(path), always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
        if sr != SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=SR)
        return y.astype(np.float32)
    except Exception as e:
        print(f'  WARN {Path(path).name}: {e}'); return None

@torch.no_grad()
def embed_whole(y):
    '''Mean-pool full audio → 768-dim.'''
    inp = _fe(y, sampling_rate=SR, return_tensors='pt', padding=False)
    out = _model(inp.input_values.to(DEVICE))
    return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

def get_windows(y):
    ws = int(WINDOW_SEC * SR)
    hs = int(HOP_SEC    * SR)
    ms = int(MIN_WIN_SEC * SR)
    if len(y) < ms: return [y]   # very short clip: use as-is
    wins, start = [], 0
    while start + ws <= len(y):
        wins.append(y[start:start+ws]); start += hs
    tail = y[start:]
    if len(tail) >= ms: wins.append(tail)
    return wins if wins else [y]

@torch.no_grad()
def embed_seg(y):
    '''10-second windows → per-window 768-dim → mean+std → 1536-dim.'''
    wins = get_windows(y)
    embs = []
    for w in wins:
        inp = _fe(w, sampling_rate=SR, return_tensors='pt', padding=False)
        e   = _model(inp.input_values.to(DEVICE)).last_hidden_state.mean(1).squeeze(0).cpu().numpy()
        embs.append(e)
    A = np.array(embs)                           # [N_windows, 768]
    return np.concatenate([A.mean(0), A.std(0)]) # [1536]


## 3a. Extract Whole-Audio Embeddings — 768-dim

Saves `{folder}_wavlm_whole.csv`. Skipped if already exists.


In [ ]:
import shutil as _shutil

for meta in folders:
    if meta['whole_csv'].exists():
        print(f"  {meta['name']}: whole_csv exists, skipping."); continue
    # Reuse v3's _wavlm.csv if it exists — no need to re-extract
    v3_csv = NB_DIR / f"{meta['name']}_wavlm.csv"
    if v3_csv.exists():
        _shutil.copy(str(v3_csv), str(meta['whole_csv']))
        print(f"  {meta['name']}: reused v3 file ({v3_csv.name} -> {meta['whole_csv'].name})"); continue
    # Fresh extraction
    print(f"
Extracting whole-audio: {meta['name']} ({len(meta['audio_files'])} files)...")
    rows = []
    for fp in tqdm(meta['audio_files'], desc=meta['name']):
        y = load_audio_16k(fp)
        if y is None: continue
        e = embed_whole(y)
        row = {'filename': fp.name}
        for i, v in enumerate(e): row[f'wavlm_{i}'] = round(float(v), 6)
        rows.append(row)
    pd.DataFrame(rows).to_csv(meta['whole_csv'], index=False)
    print(f"  Saved {len(rows)} rows -> {meta['whole_csv'].name}")


## 3b. Extract Segmented Embeddings — 1536-dim

Saves `{folder}_wavlm_seg.csv`. Skipped if already exists.


In [ ]:
for meta in folders:
    if meta['seg_csv'].exists():
        print(f"  {meta['name']}: seg_csv exists, skipping."); continue
    print(f"\nExtracting segmented: {meta['name']} ({len(meta['audio_files'])} files)...")
    rows = []
    for fp in tqdm(meta['audio_files'], desc=meta['name']):
        y = load_audio_16k(fp)
        if y is None: continue
        e = embed_seg(y)
        row = {'filename': fp.name}
        for i, v in enumerate(e[:768]):  row[f'wavlm_mean_{i}'] = round(float(v), 6)
        for i, v in enumerate(e[768:]):  row[f'wavlm_std_{i}']  = round(float(v), 6)
        rows.append(row)
    pd.DataFrame(rows).to_csv(meta['seg_csv'], index=False)
    print(f"  Saved {len(rows)} rows  ({len(rows[0])-1} features) → {meta['seg_csv'].name}")


## 4. Load GT Labels + Build DataFrames


In [ ]:
def load_folder_df(meta, feat_type):
    csv_key = 'whole_csv' if feat_type == 'whole' else 'seg_csv'
    feat_df = pd.read_csv(meta[csv_key])

    gt      = pd.read_csv(meta['gt_path'])
    fn_col  = next((c for c in gt.columns if c.lower() in ('filename','file','name')), gt.columns[0])
    lbl_col = next((c for c in gt.columns if c.lower() in ('label','class','cheating','gt')), gt.columns[-1])
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].astype(str).str.lower().str.strip().map(LABEL_MAP)
    gt = gt.dropna(subset=['label_int'])
    gt['label_int'] = gt['label_int'].astype(int)

    merged = feat_df.merge(gt[['filename','label_int']], on='filename', how='inner')
    merged['audio_batch'] = meta['name']
    return merged

def build_combined(folder_names, feat_type):
    dfs = []
    for name in folder_names:
        meta = next((m for m in folders if m['name'] == name), None)
        if meta: dfs.append(load_folder_df(meta, feat_type))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

train_names = TRAIN_FOLDERS
test_names  = [TEST_FOLDER] if TEST_FOLDER else []

train_whole = build_combined(train_names, 'whole')
train_seg   = build_combined(train_names, 'seg')
test_whole  = build_combined(test_names,  'whole') if test_names else pd.DataFrame()
test_seg    = build_combined(test_names,  'seg')   if test_names else pd.DataFrame()

print(f'Train: {len(train_whole)} samples  '
      f'(cheating={int((train_whole.label_int==1).sum())}  '
      f'not_cheating={int((train_whole.label_int==0).sum())})')
if not test_whole.empty:
    print(f'Test:  {len(test_whole)} samples from {TEST_FOLDER}  '
          f'(cheating={int((test_whole.label_int==1).sum())}  '
          f'not_cheating={int((test_whole.label_int==0).sum())})')
else:
    print(f'Test:  will use {int(TEST_RATIO*100)}% of train (no TEST_FOLDER set)')


## 5. Train / Test Split

- `TEST_FOLDER` set → that folder is test, all `TRAIN_FOLDERS` are train (group-wise).
- `TEST_FOLDER` empty → stratified 80/20 split of combined train data.


In [ ]:
feat_whole_cols = [c for c in train_whole.columns if c.startswith('wavlm_')]
feat_seg_cols   = [c for c in train_seg.columns   if c.startswith('wavlm_')]

if not test_whole.empty:
    X_tr_whole = train_whole[feat_whole_cols].fillna(0).values
    X_te_whole = test_whole[feat_whole_cols].fillna(0).values
    X_tr_seg   = train_seg[feat_seg_cols].fillna(0).values
    X_te_seg   = test_seg[feat_seg_cols].fillna(0).values
    y_tr = train_whole['label_int'].values
    y_te = test_whole['label_int'].values
    print(f'Group-wise — train={len(y_tr)}, test={len(y_te)}')
else:
    y_all = train_whole['label_int'].values
    idx   = np.arange(len(y_all))
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_RATIO,
                                      random_state=RANDOM_SEED, stratify=y_all)
    X_tr_whole = train_whole.iloc[tr_idx][feat_whole_cols].fillna(0).values
    X_te_whole = train_whole.iloc[te_idx][feat_whole_cols].fillna(0).values
    X_tr_seg   = train_seg.iloc[tr_idx][feat_seg_cols].fillna(0).values
    X_te_seg   = train_seg.iloc[te_idx][feat_seg_cols].fillna(0).values
    y_tr = y_all[tr_idx]
    y_te = y_all[te_idx]
    print(f'Random split — train={len(y_tr)}, test={len(y_te)}')

spw = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1.0)
print(f'scale_pos_weight = {spw:.2f}')
print(f'Train: {int((y_tr==1).sum())} cheating / {int((y_tr==0).sum())} not')
print(f'Test:  {int((y_te==1).sum())} cheating / {int((y_te==0).sum())} not')


## 6. Train & Evaluate


In [ ]:
def train_xgb(X_tr, X_te, y_tr, y_te, label, spw):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_tr)
    Xte = scaler.transform(X_te)

    # Lower colsample for high-dimensional feature spaces
    colsample = 0.3 if X_tr.shape[1] > 500 else 0.8
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.04,
        subsample=0.8, colsample_bytree=colsample,
        min_child_weight=3, scale_pos_weight=spw,
        eval_metric='logloss', early_stopping_rounds=30,
        random_state=RANDOM_SEED, device='cpu',
    )
    model.fit(Xtr, y_tr, eval_set=[(Xte, y_te)], verbose=False)
    proba = model.predict_proba(Xte)[:, 1]

    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f1 = f1_score(y_te, (proba >= thr).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thr = f1, thr

    preds = (proba >= best_thr).astype(int)
    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'  Features: {X_tr.shape[1]}-dim   Best threshold: {best_thr:.2f}')
    print(f'{"="*55}')
    print(classification_report(y_te, preds,
          target_names=['not_cheating','cheating'], digits=4, zero_division=0))
    cm = confusion_matrix(y_te, preds, labels=[0,1])
    print(f'  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')
    return dict(
        label=label, n_feats=X_tr.shape[1],
        f1=round(best_f1,4), threshold=round(best_thr,2),
        precision=round(precision_score(y_te,preds,zero_division=0),4),
        recall=round(recall_score(y_te,preds,zero_division=0),4),
        accuracy=round(accuracy_score(y_te,preds),4),
        model=model, scaler=scaler, proba=proba, preds=preds,
    )


In [ ]:
res_whole = train_xgb(X_tr_whole, X_te_whole, y_tr, y_te,
                      label='Whole-audio 768-dim (baseline)', spw=spw)

res_seg   = train_xgb(X_tr_seg, X_te_seg, y_tr, y_te,
                      label='Segmented 1536-dim  (mean+std)', spw=spw)


In [ ]:
# ── Fine threshold sweep: 0.30 .. 0.70 step 0.01 ────────────────────────────
# Prints one row per threshold for each variant so you can see exactly where
# precision/recall flip and diagnose why the two notebooks give different results.

def fine_sweep(proba, y, lo=0.30, hi=0.70, step=0.01):
    rows = []
    for thr in np.arange(lo, hi + 1e-9, step):
        pred = (proba >= thr).astype(int)
        cm   = confusion_matrix(y, pred, labels=[0, 1])
        tn, fp, fn_, tp = int(cm[0,0]), int(cm[0,1]), int(cm[1,0]), int(cm[1,1])
        rows.append(dict(
            thr=round(float(thr), 2),
            prec=round(precision_score(y, pred, zero_division=0), 4),
            rec=round(recall_score(y, pred, zero_division=0), 4),
            f1=round(f1_score(y, pred, zero_division=0), 4),
            tp=tp, fp=fp, fn=fn_, tn=tn,
            pred_pos=tp + fp,
        ))
    return pd.DataFrame(rows)

sweep_whole = fine_sweep(res_whole['proba'], y_te)
sweep_seg   = fine_sweep(res_seg['proba'],   y_te)

sweep_whole.to_csv(SAVE_DIR / 'sweep_whole.csv', index=False)
sweep_seg.to_csv(SAVE_DIR / 'sweep_seg.csv',   index=False)

print('=' * 78)
print('  FINE THRESHOLD SWEEP  (0.30 -> 0.70 step 0.01)')
print(f'  Test pos={int((y_te==1).sum())}  neg={int((y_te==0).sum())}  total={len(y_te)}')
print('=' * 78)

for label, sw in [('WHOLE 768-dim', sweep_whole), ('SEGMENTED 1536-dim', sweep_seg)]:
    print(f'\n-- {label} --')
    print(sw.to_string(index=False))
    best = sw.loc[sw['f1'].idxmax()]
    print(f"  best F1 @ thr={best['thr']:.2f}: f1={best['f1']:.4f} "
          f"prec={best['prec']:.4f} rec={best['rec']:.4f} "
          f"tp={int(best['tp'])} fp={int(best['fp'])} fn={int(best['fn'])}")

# Side-by-side diff on the shared threshold grid
diff = sweep_whole.merge(sweep_seg, on='thr', suffixes=('_w', '_s'))
diff['f1_delta']   = (diff['f1_s']   - diff['f1_w']).round(4)
diff['prec_delta'] = (diff['prec_s'] - diff['prec_w']).round(4)
diff['rec_delta']  = (diff['rec_s']  - diff['rec_w']).round(4)
cols = ['thr', 'f1_w', 'f1_s', 'f1_delta', 'prec_w', 'prec_s', 'prec_delta',
        'rec_w', 'rec_s', 'rec_delta', 'tp_w', 'tp_s', 'fp_w', 'fp_s']
print('\n' + '=' * 78)
print('  SIDE-BY-SIDE  (segmented - whole)')
print('=' * 78)
print(diff[cols].to_string(index=False))
diff[cols].to_csv(SAVE_DIR / 'sweep_diff_whole_vs_seg.csv', index=False)
print(f"\nSaved -> {SAVE_DIR/'sweep_whole.csv'},  {SAVE_DIR/'sweep_seg.csv'},  {SAVE_DIR/'sweep_diff_whole_vs_seg.csv'}")

## 7. Comparison


In [ ]:
print(f'\n{"="*60}')
print('  COMPARISON SUMMARY')
print(f'{"="*60}')
hdr = f"  {'Model':<34} {'F1':>6} {'Prec':>6} {'Rec':>6} {'Acc':>6} {'Thr':>5} {'Feats':>6}"
print(hdr)
print(f"  {'-'*34} {'-'*6} {'-'*6} {'-'*6} {'-'*6} {'-'*5} {'-'*6}")
for r in [res_whole, res_seg]:
    print(f"  {r['label']:<34} {r['f1']:>6.4f} {r['precision']:>6.4f} "
          f"{r['recall']:>6.4f} {r['accuracy']:>6.4f} {r['threshold']:>5.2f} {r['n_feats']:>6}")
print(f'{"="*60}')
delta = res_seg['f1'] - res_whole['f1']
sign  = '+' if delta >= 0 else ''
print(f'  F1 delta (segmented - whole): {sign}{delta:.4f}')
print()
if delta > 0.02:
    print('  Result: temporal consistency signal IS real.')
    print('  Next step: fine-tune WavLM (Stage B) for even better embeddings.')
elif delta < -0.02:
    print('  Result: whole-audio is better. Check if audio durations are very short.')
    print('  Try: WINDOW_SEC=5, HOP_SEC=2 and re-run from cell 3b.')
else:
    print('  Result: similar. Fine-tuning WavLM will be the deciding factor.')


In [ ]:

# ── Plain-English Result Interpreter ─────────────────────────────────────────
# Run this cell after the comparison table to understand what the numbers mean.

def interpret_results(res_whole, res_seg, y_te):
    n_test       = len(y_te)
    n_cheating   = int((y_te == 1).sum())
    n_honest     = int((y_te == 0).sum())

    print("=" * 62)
    print("  WHAT THE NUMBERS MEAN  (cheating detection context)")
    print("=" * 62)
    print(f"\n  Test set: {n_test} students  "
          f"({n_cheating} cheating  /  {n_honest} honest)\n")

    for res in [res_whole, res_seg]:
        prec = res['precision']
        rec  = res['recall']
        thr  = res['threshold']
        f1   = res['f1']

        # Reconstruct confusion matrix numbers from proba
        preds = (res['proba'] >= thr).astype(int)
        tp = int(((preds == 1) & (y_te == 1)).sum())
        fp = int(((preds == 1) & (y_te == 0)).sum())
        fn = int(((preds == 0) & (y_te == 1)).sum())
        tn = int(((preds == 0) & (y_te == 0)).sum())
        flagged = tp + fp

        print(f"  ── {res['label']} ──")
        print(f"     Threshold used: {thr:.2f}  "
              f"(model calls 'cheating' when confidence > {int(thr*100)}%)")
        print(f"     Students flagged as cheating: {flagged} / {n_test}")
        print()
        print(f"     Precision {prec:.0%}  →  of {flagged} flagged, "
              f"{tp} were real cheaters, {fp} were innocent (false alarms)")
        print(f"     Recall    {rec:.0%}  →  of {n_cheating} actual cheaters, "
              f"caught {tp}, missed {fn}")
        print(f"     F1        {f1:.4f}  →  combined score (1.0 = perfect)")
        print()
        print(f"     In plain English:")
        if prec >= 0.70:
            print(f"       When it flags someone, it's usually right ({prec:.0%} of the time).")
        else:
            print(f"       Fairly noisy — only right {prec:.0%} of the time it flags someone.")
        if rec >= 0.65:
            print(f"       Catches most cheaters ({rec:.0%} of them).")
        else:
            print(f"       Misses a lot of cheaters — only catches {rec:.0%} of them.")
        print()

    # ── Head-to-head verdict ─────────────────────────────────────────────────
    print("=" * 62)
    print("  WHICH MODEL TO USE?")
    print("=" * 62)
    p_w, r_w = res_whole['precision'], res_whole['recall']
    p_s, r_s = res_seg['precision'],   res_seg['recall']

    print(f"""
  Whole-audio  →  higher precision ({p_w:.0%}), lower recall ({r_w:.0%})
    Fewer false alarms, but misses more actual cheaters.
    Use when: wrongly accusing an honest student is the bigger risk.

  Segmented    →  lower precision ({p_s:.0%}), higher recall ({r_s:.0%})
    Catches more cheaters, but flags more innocent students too.
    Use when: missing a cheater is the bigger risk.

  For Mettl's use case (exam integrity):
    → Segmented is likely better RIGHT NOW because catching more cheaters
      is the primary goal, and flagged cases are reviewed by a human anyway
      so false alarms get filtered before any action is taken.

  BUT both numbers will improve significantly after WavLM fine-tuning
  (Stage B). These results use only the pretrained baseline embeddings.
  Run this notebook again with WAVLM_MODEL_PATH pointing to the
  fine-tuned model once training on Kaggle finishes.
""")

interpret_results(res_whole, res_seg, y_te)


## 8. Save Models & Results


In [ ]:
for res, tag in [(res_whole, 'whole'), (res_seg, 'seg')]:
    res['model'].save_model(str(SAVE_DIR / f'xgb_{tag}.json'))
    joblib.dump(res['scaler'], str(SAVE_DIR / f'scaler_{tag}.pkl'))

skip = {'model', 'scaler', 'proba', 'preds'}
out  = {
    'wavlm_model':   WAVLM_MODEL_PATH,
    'window_sec':    WINDOW_SEC,
    'hop_sec':       HOP_SEC,
    'train_folders': TRAIN_FOLDERS,
    'test_folder':   TEST_FOLDER,
    'whole': {k: v for k, v in res_whole.items() if k not in skip},
    'seg':   {k: v for k, v in res_seg.items()   if k not in skip},
}
with open(SAVE_DIR / 'results.json', 'w') as f:
    _json.dump(out, f, indent=2)

print(f'Saved to {SAVE_DIR}/')
print('  xgb_whole.json + scaler_whole.pkl')
print('  xgb_seg.json   + scaler_seg.pkl')
print('  results.json')
